In [0]:
from pyspark.sql import functions as F

# Step 1: load the raw data
bronze = spark.table("carbon_intensity.bronze.intensity_raw")

# Step 2: tell Spark the shape of the JSON so it can read it
schema = "struct<data:array<struct<from:string,to:string,intensity:struct<forecast:int,actual:int,index:string>>>>"
bronze = bronze.withColumn("parsed", F.from_json("raw_json", schema))

# Step 3: each API pull has multiple time periods inside it - split them into separate rows
bronze = bronze.select(F.explode("parsed.data").alias("d"), "ingested_at")

# Step 4: pull out the fields we actually want, with simple names
silver = bronze.select(
    F.to_timestamp("d.from", "yyyy-MM-dd'T'HH:mm'Z'").alias("period_start"),
    F.col("d.intensity.forecast").alias("forecast"),
    F.col("d.intensity.actual").alias("actual"),
    F.col("d.intensity.index").alias("intensity_band")
)
silver.show(5)

In [0]:
# Step 5: use actual if we have it, otherwise use forecast
silver = silver.withColumn("intensity_value", F.coalesce("actual", "forecast"))

silver.show(5)

In [0]:
# Step 6: pull out the hour and day, since that's what we need to answer "greenest time"
silver = silver.withColumn("hour_of_day", F.hour("period_start"))
silver = silver.withColumn("day_of_week", F.date_format("period_start", "E"))

silver.show(5)

In [0]:
# Step 7: remove exact duplicate rows (simple version - not perfect, but easy to read)
silver = silver.dropDuplicates(["period_start"])
silver.show(5)


In [0]:

# Step 8: save it
silver.write.mode("overwrite").saveAsTable("carbon_intensity.silver.intensity_clean")
silver.show(5)

In [0]:
# Step 9: check it looks right
print("Row count:", silver.count())


In [0]:
from pyspark.sql import functions as F
# Imports Spark's function library (F.col, F.to_timestamp, F.hour, etc.) - the toolkit for this notebook

bronze = spark.table("carbon_intensity.bronze.intensity_raw")
# Loads the entire Bronze table into memory as a DataFrame - every row ever ingested, duplicates included

schema = "struct<data:array<struct<from:string,to:string,intensity:struct<forecast:int,actual:int,index:string>>>>"
# Describes the exact shape of the API's JSON so Spark knows how to read it:
# a "data" array, where each item has from/to timestamps (as text) and an "intensity" object
# containing forecast (int), actual (int), index (text)

silver = (bronze
    .withColumn("parsed", F.from_json("raw_json", schema))
    # Converts the raw_json TEXT column into real structured data, using the schema above.
    # Before this line, raw_json is just a string; after, "parsed" is a real nested object Spark can query.

    .select(F.explode("parsed.data").alias("d"), "ingested_at")
    # "parsed.data" is an array (one item per 30-min period). explode() turns that array into
    # separate ROWS - one row per period, instead of one row per API call.
    # We keep "ingested_at" alongside so we still know when each row was pulled.

    .select(
        F.to_timestamp("d.from", "yyyy-MM-dd'T'HH:mm'Z'").alias("period_start"),
        # Pulls the "from" field out of the nested struct "d", converts it from text to a real
        # timestamp type using the exact format your API returns, names it "period_start"

        F.to_timestamp("d.to", "yyyy-MM-dd'T'HH:mm'Z'").alias("period_end"),
        # Same, for the "to" field -> "period_end"

        F.col("d.intensity.forecast").alias("forecast"),
        # Pulls the forecast number out of the nested "intensity" struct, flattens it into its own column

        F.col("d.intensity.actual").alias("actual"),
        # Same for "actual" - will be NULL for periods that haven't settled yet

        F.col("d.intensity.index").alias("intensity_band"),
        # Pulls the text label ("low"/"moderate"/"high" etc.) into its own column

        "ingested_at"
        # Carries this column forward into the new selection (it would be dropped otherwise)
    )

    .withColumn("intensity_value", F.coalesce("actual", "forecast"))
    # Creates ONE clean number to analyse: uses "actual" if it exists, otherwise falls back to "forecast".
    # This is what solves the null-actual issue you saw earlier.

    .withColumn("hour_of_day", F.hour("period_start"))
    # Extracts just the hour (0-23) from the timestamp - this is what "greenest hour" analysis groups by

    .withColumn("day_of_week", F.date_format("period_start", "E"))
    # Extracts the day name abbreviation (Mon, Tue, etc.) from the timestamp

    .withColumn("is_weekend", F.dayofweek("period_start").isin([1, 7]))
    # True/False flag - Spark's dayofweek() returns 1=Sunday, 7=Saturday, so this flags weekends

    .withColumn("rn", F.row_number().over(
        __import__("pyspark").sql.window.Window
            .partitionBy("period_start")
            .orderBy(F.col("ingested_at").desc())
    ))
    # For each period_start, ranks all its rows from newest ingested_at (1) to oldest.
    # This is how we identify duplicates - if a period was pulled 5 times, this numbers them 1-5.

    .filter("rn = 1")
    # Keeps ONLY the most recent pull (rank 1) for each period_start - discards older duplicate pulls

    .drop("rn")
    # Removes the helper ranking column now that it's done its job - it's not meant to be in the final table
)

silver.write.mode("overwrite").saveAsTable("carbon_intensity.silver.intensity_clean")
# Saves the cleaned result as a table. mode("overwrite") replaces the whole table each run -
# unlike Bronze's append, Silver is always rebuilt fresh from all of Bronze, so it can never drift out of sync

print(f"Rows: {silver.count()}")
# Prints total row count - your basic sanity check that data exists and roughly matches expectations

print(f"Null intensity_value count: {silver.filter(F.col('intensity_value').isNull()).count()}")
# Counts rows where BOTH actual and forecast were somehow missing - should be 0 or very low.
# A high number here would mean something's wrong with the API data or the parsing.

silver.select("period_start", "intensity_value", "intensity_band", "hour_of_day", "day_of_week").show(5)
# Prints 5 sample rows so you can visually eyeball that the data looks sensible before trusting it